# Bronze Ingest: Citi Bike Trip Data

Downloads a monthly Citi Bike (NYC) trip-data zip, extracts its CSV(s), and
persists them as a managed Spark table, unmodified from source. This is the
`bronze` layer.

Re-runnable: uses `CREATE DATABASE IF NOT EXISTS` and `mode("overwrite")`,
so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Configuration

Change `MONTH` to pull a different month without touching the logic below.
Available months are listed at https://s3.amazonaws.com/tripdata/index.html

In [ ]:
DATABASE_NAME = "citibike"
TABLE_NAME = "tripdata"
MONTH = "202501"
SOURCE_URL = f"https://s3.amazonaws.com/tripdata/{MONTH}-citibike-tripdata.zip"
LOCAL_ZIP = f"/tmp/{MONTH}-citibike-tripdata.zip"
LOCAL_DIR = f"/tmp/citibike_{MONTH}"

## Download source file

In [ ]:
import urllib.request

urllib.request.urlretrieve(SOURCE_URL, LOCAL_ZIP)

## Extract CSV(s)

The zip may contain one CSV or several (split by borough/size); macOS
metadata entries (`__MACOSX`) are filtered out.

In [ ]:
import glob
import zipfile

with zipfile.ZipFile(LOCAL_ZIP) as zf:
    zf.extractall(LOCAL_DIR)

csv_files = [
    f for f in glob.glob(f"{LOCAL_DIR}/**/*.csv", recursive=True)
    if "__MACOSX" not in f
]
csv_files

## Load dataset

In [ ]:
df = spark.read.csv(csv_files, header=True, inferSchema=True)

## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'Citi Bike (NYC) trip data used for analytics demonstrations'
""")

## Persist as a managed Spark table

In [ ]:
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_rides FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)